# Признаки и их вклад

Запускать из корня проекта после main.py. Ячейки последовательно воспроизводят абляции, проверку новых признаков и график зависимости доли ботов от значения признака.


In [1]:
from pathlib import Path
import os

if not Path('main.py').exists() and (Path.cwd().parent / 'main.py').exists():
    os.chdir(Path.cwd().parent)


## Вклад групп признаков


In [2]:
import json
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from main import FEATURES, adjust_pair, new_char_vectorizer, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"
GROUPS = {
    "conversation_geometry": [
        "n_msg", "other_n", "n_diff", "first_speaker", "first_turn_fraction",
        "last_turn_fraction", "is_last_speaker", "consecutive_own_turns",
    ],
    "reply_copy": [
        "shared_vocab_ratio", "avg_prev_overlap", "max_prev_overlap",
        "avg_prev_similarity", "max_prev_similarity", "exact_echo_share",
        "contains_echo_share", "avg_log_reply_len_ratio",
        "replied_to_question_share", "avg_self_adjacent_similarity",
    ],
    "length_lexical": [
        "chars", "avg_msg_chars", "max_msg_chars", "std_msg_chars",
        "words", "unique_words", "char_diff", "median_msg_chars",
        "q25_msg_chars", "q75_msg_chars", "first_msg_chars",
        "last_msg_chars", "avg_msg_words", "max_msg_words", "unique_msg",
        "char_entropy", "word_entropy", "long_msg_share", "empty_msg_share",
    ],
    "style_script": [
        "punct", "digit", "uppercase", "cyrillic", "latin", "newline",
        "parentheses", "question_msg_share", "exclaim_msg_share",
        "cyrillic_msg_share", "latin_msg_share", "whitespace_ratio",
        "supplementary_unicode_ratio", "question_char_ratio",
        "exclaim_char_ratio",
    ],
}


def candidate(fold: int) -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=600, depth=5, learning_rate=.04, l2_leaf_reg=4,
        loss_function="Logloss", random_seed=42 + fold,
        thread_count=4, verbose=False,
    )


def log_study(name: str, raw: float, paired: float, train_score: float | None) -> None:
    with mlflow.start_run(run_name=f"ablation_{name}"):
        mlflow.log_params({"study": "feature_ablation", "scenario": name, "folds": 4, "seed": 23})
        mlflow.log_metric("oof_logloss", raw)
        mlflow.log_metric("oof_logloss_pair_adjusted", paired)
        if train_score is not None:
            mlflow.log_metric("mean_fold_train_logloss", train_score)


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    names = [feature for group in GROUPS.values() for feature in group]
    if len(names) != len(FEATURES) or set(names) != set(FEATURES):
        raise ValueError("Feature groups must partition all numeric features")

    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    labels = train["is_bot"].to_numpy()
    numeric = train.select(FEATURES).to_numpy().astype(float)
    texts = train["text"].to_list()
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    scenarios = {"full": list(range(len(FEATURES))), "text_svd_only": []}
    for group, excluded in GROUPS.items():
        scenarios[f"without_{group}"] = [i for i, feature in enumerate(FEATURES) if feature not in excluded]

    oof = {name: np.zeros(len(train)) for name in scenarios}
    train_losses = {name: [] for name in scenarios}
    permutation_losses = {name: [] for name in FEATURES + ["text_svd"]}
    group_permutation_losses = {name: [] for name in GROUPS}
    cv = StratifiedKFold(4, shuffle=True, random_state=23)
    for fold, (train_dialogues, val_dialogues) in enumerate(
        cv.split(np.arange(len(pair_classes)), pair_classes)
    ):
        tr = row_indices(train_dialogues)
        va = row_indices(val_dialogues)
        vectorizer = new_char_vectorizer()
        x_train = vectorizer.fit_transform([texts[i] for i in tr])
        x_val = vectorizer.transform([texts[i] for i in va])
        svd = TruncatedSVD(n_components=48, n_iter=5, random_state=42 + fold)
        train_svd = svd.fit_transform(x_train)
        val_svd = svd.transform(x_val)
        for name, indices in scenarios.items():
            fit_x = np.hstack([numeric[tr][:, indices], train_svd])
            val_x = np.hstack([numeric[va][:, indices], val_svd])
            model = candidate(fold)
            model.fit(fit_x, labels[tr])
            oof[name][va] = model.predict_proba(val_x)[:, 1]
            train_losses[name].append(float(log_loss(labels[tr], model.predict_proba(fit_x)[:, 1])))
            if name != "full":
                continue
            baseline_loss = float(log_loss(labels[va], oof[name][va]))
            for feature_index, feature in enumerate(FEATURES):
                shuffled = val_x.copy()
                rng = np.random.default_rng(1000 + 53 * fold + feature_index)
                shuffled[:, feature_index] = shuffled[rng.permutation(len(va)), feature_index]
                loss = float(log_loss(labels[va], model.predict_proba(shuffled)[:, 1]))
                permutation_losses[feature].append((loss - baseline_loss, len(va)))
            shuffled = val_x.copy()
            rng = np.random.default_rng(5000 + fold)
            shuffled[:, len(FEATURES):] = shuffled[rng.permutation(len(va)), len(FEATURES):]
            loss = float(log_loss(labels[va], model.predict_proba(shuffled)[:, 1]))
            permutation_losses["text_svd"].append((loss - baseline_loss, len(va)))
            for group, group_features in GROUPS.items():
                shuffled = val_x.copy()
                positions = [FEATURES.index(feature) for feature in group_features]
                rng = np.random.default_rng(7000 + fold + len(positions))
                shuffled[:, positions] = shuffled[rng.permutation(len(va))][:, positions]
                loss = float(log_loss(labels[va], model.predict_proba(shuffled)[:, 1]))
                group_permutation_losses[group].append((loss - baseline_loss, len(va)))
        print(f"fold {fold + 1}/4 done", flush=True)

    scores = {
        name: {
            "raw": round(float(log_loss(labels, probabilities)), 6),
            "pair_adjusted": round(float(log_loss(labels, adjust_pair(probabilities))), 6),
            "mean_fold_train": round(float(np.mean(train_losses[name])), 6),
        }
        for name, probabilities in oof.items()
    }
    reference = scores["full"]["pair_adjusted"]
    for name, item in scores.items():
        item["delta_vs_full_pair"] = round(item["pair_adjusted"] - reference, 6)

    def weighted_delta(items):
        return round(sum(value * count for value, count in items) / sum(count for _, count in items), 6)

    report = {
        "groups": GROUPS,
        "scores": scores,
        "group_permutation_delta": {
            name: weighted_delta(items) for name, items in group_permutation_losses.items()
        },
        "individual_permutation_delta": dict(sorted(
            ((name, weighted_delta(items)) for name, items in permutation_losses.items()),
            key=lambda item: -item[1],
        )),
    }
    path = OUTPUTS / "analysis/feature_ablation.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    np.savez(OUTPUTS / "analysis/feature_ablation_oof.npz", **oof)
    database = (OUTPUTS / "mlflow.db").resolve()
    mlflow.set_tracking_uri(f"sqlite:///{database.as_posix()}")
    mlflow.set_experiment("bot_detection")
    for name, item in scores.items():
        log_study(name, item["raw"], item["pair_adjusted"], item["mean_fold_train"])
    print(json.dumps({
        "scores": scores,
        "group_permutation_delta": report["group_permutation_delta"],
        "top_individual": list(report["individual_permutation_delta"].items())[:12],
    }, ensure_ascii=False, indent=2))

main()


fold 1/4 done
fold 2/4 done
fold 3/4 done
fold 4/4 done
{
  "scores": {
    "full": {
      "raw": 0.184413,
      "pair_adjusted": 0.180772,
      "mean_fold_train": 0.048393,
      "delta_vs_full_pair": 0.0
    },
    "text_svd_only": {
      "raw": 0.507318,
      "pair_adjusted": 0.47297,
      "mean_fold_train": 0.262746,
      "delta_vs_full_pair": 0.292198
    },
    "without_conversation_geometry": {
      "raw": 0.226254,
      "pair_adjusted": 0.221071,
      "mean_fold_train": 0.060989,
      "delta_vs_full_pair": 0.040299
    },
    "without_reply_copy": {
      "raw": 0.252446,
      "pair_adjusted": 0.253526,
      "mean_fold_train": 0.085982,
      "delta_vs_full_pair": 0.072754
    },
    "without_length_lexical": {
      "raw": 0.185784,
      "pair_adjusted": 0.181684,
      "mean_fold_train": 0.047725,
      "delta_vs_full_pair": 0.000912
    },
    "without_style_script": {
      "raw": 0.204227,
      "pair_adjusted": 0.202854,
      "mean_fold_train": 0.051636,
  

In [3]:
import json
from pathlib import Path

import numpy as np
import polars as pl

from main import adjust_pair


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"


def dialogue_losses(labels: np.ndarray, probabilities: np.ndarray) -> np.ndarray:
    adjusted = np.clip(adjust_pair(probabilities), 1e-8, 1 - 1e-8)
    losses = -labels * np.log(adjusted) - (1 - labels) * np.log1p(-adjusted)
    return losses.reshape(-1, 2).mean(axis=1)


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    labels = train["is_bot"].to_numpy()
    predictions = np.load(OUTPUTS / "analysis/feature_ablation_oof.npz")
    full = dialogue_losses(labels, predictions["full"])
    rng = np.random.default_rng(2026)
    sample_indices = rng.integers(0, len(full), size=(5000, len(full)))
    results = {}
    for name in predictions.files:
        if name == "full":
            continue
        differences = dialogue_losses(labels, predictions[name]) - full
        bootstraps = differences[sample_indices].mean(axis=1)
        results[name] = {
            "delta": round(float(differences.mean()), 6),
            "bootstrap_95pct": [round(float(value), 6) for value in np.quantile(bootstraps, [.025, .975])],
            "bootstrap_positive_share": round(float(np.mean(bootstraps > 0)), 4),
        }
    path = OUTPUTS / "analysis/feature_ablation_uncertainty.json"
    path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(results, ensure_ascii=False, indent=2))

main()


{
  "text_svd_only": {
    "delta": 0.292199,
    "bootstrap_95pct": [
      0.266788,
      0.318365
    ],
    "bootstrap_positive_share": 1.0
  },
  "without_conversation_geometry": {
    "delta": 0.0403,
    "bootstrap_95pct": [
      0.024044,
      0.057691
    ],
    "bootstrap_positive_share": 1.0
  },
  "without_reply_copy": {
    "delta": 0.072754,
    "bootstrap_95pct": [
      0.054781,
      0.091401
    ],
    "bootstrap_positive_share": 1.0
  },
  "without_length_lexical": {
    "delta": 0.000912,
    "bootstrap_95pct": [
      -0.004765,
      0.006912
    ],
    "bootstrap_positive_share": 0.6362
  },
  "without_style_script": {
    "delta": 0.022082,
    "bootstrap_95pct": [
      0.014493,
      0.030154
    ],
    "bootstrap_positive_share": 1.0
  }
}


In [4]:
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

report = json.loads(Path('outputs/analysis/feature_ablation.json').read_text(encoding='utf-8'))
names = ['without_reply_copy', 'without_conversation_geometry', 'without_style_script', 'without_length_lexical']
labels = ['Ответы и повторы', 'Порядок ходов', 'Стиль', 'Длина и словарь']
values = [report['scores'][name]['delta_vs_full_pair'] for name in names]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(labels[::-1], values[::-1], color='#7055F4')
ax.set_xlabel('Рост OOF LogLoss при удалении группы')
ax.grid(axis='x', alpha=.2)
fig.tight_layout()
fig.savefig('outputs/analysis/feature_ablation_plot.png', dpi=160)
plt.show()


02_features.ipynb:cell5:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


![Вклад групп признаков](../outputs/analysis/feature_ablation_plot.png)

Сильнее всего качество ухудшилось без признаков реакции на собеседника. Удаление длины и словарного разнообразия почти не изменило результат. Это вклад групп в данной модели, а не причинный эффект отдельных признаков.


## Новые признаки реакции


In [5]:
import json
import re
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from main import FEATURES, adjust_pair, new_char_vectorizer, read_dialogues, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"
NEW_FEATURES = [
    "reply_overlap_high_share", "reply_novel_word_share", "reply_longer_share",
    "reply_prefix_extension_share", "reply_prefix_truncation_share",
    "reply_common_prefix_share", "reply_common_suffix_share",
    "last_reply_overlap", "last_reply_length_ratio",
    "question_to_question_share", "reply_script_switch_share",
]


def tokens(text: str) -> set[str]:
    return set(re.findall(r"\w+", text.casefold()))


def average(values) -> float:
    return float(np.mean(values)) if values else 0.0


def common_prefix(a: str, b: str) -> int:
    count = 0
    for x, y in zip(a, b):
        if x != y:
            break
        count += 1
    return count


def new_features(messages, index: int) -> list[float]:
    previous = None
    overlaps = []
    novelty = []
    longer = []
    extension = []
    truncation = []
    prefix = []
    suffix = []
    ratios = []
    question_to_question = []
    script_switch = []
    for message in messages:
        if message.participant_index != index:
            previous = message.text.strip().casefold()
            continue
        if previous is None:
            continue
        current = message.text.strip().casefold()
        own_words = tokens(current)
        other_words = tokens(previous)
        overlaps.append(len(own_words & other_words) / max(len(own_words | other_words), 1))
        novelty.append(len(own_words - other_words) / max(len(own_words), 1))
        longer.append(int(len(current) > len(previous)))
        extension.append(int(bool(previous) and current.startswith(previous) and len(current) > len(previous)))
        truncation.append(int(bool(current) and previous.startswith(current) and len(previous) > len(current)))
        prefix.append(common_prefix(current, previous) / max(min(len(current), len(previous)), 1))
        suffix.append(common_prefix(current[::-1], previous[::-1]) / max(min(len(current), len(previous)), 1))
        ratios.append((len(current) + 1) / (len(previous) + 1))
        if "?" in previous:
            question_to_question.append(int("?" in current))
        own_script = "cyr" if re.search(r"[а-я]", current) else "lat" if re.search(r"[a-z]", current) else "other"
        other_script = "cyr" if re.search(r"[а-я]", previous) else "lat" if re.search(r"[a-z]", previous) else "other"
        script_switch.append(int(own_script != other_script))
    return [
        average([value > .5 for value in overlaps]),
        average(novelty), average(longer), average(extension), average(truncation),
        average(prefix), average(suffix), overlaps[-1] if overlaps else 0.0,
        ratios[-1] if ratios else 0.0, average(question_to_question),
        average(script_switch),
    ]


def run_seed(seed: int, numeric: np.ndarray, extended: np.ndarray, texts: list[str], labels: np.ndarray) -> tuple[dict, np.ndarray]:
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    cv = StratifiedKFold(4, shuffle=True, random_state=seed)
    oof = np.zeros(len(labels))
    train_scores = []
    for fold, (train_dialogues, val_dialogues) in enumerate(
        cv.split(np.arange(len(pair_classes)), pair_classes)
    ):
        tr = row_indices(train_dialogues)
        va = row_indices(val_dialogues)
        vectorizer = new_char_vectorizer()
        char_train = vectorizer.fit_transform([texts[i] for i in tr])
        char_val = vectorizer.transform([texts[i] for i in va])
        svd = TruncatedSVD(n_components=48, n_iter=5, random_state=42 + fold)
        fit_x = np.hstack([numeric[tr], extended[tr], svd.fit_transform(char_train)])
        val_x = np.hstack([numeric[va], extended[va], svd.transform(char_val)])
        model = CatBoostClassifier(
            iterations=600, depth=5, learning_rate=.04, l2_leaf_reg=4,
            loss_function="Logloss", random_seed=42 + fold, thread_count=4,
            verbose=False,
        )
        model.fit(fit_x, labels[tr])
        oof[va] = model.predict_proba(val_x)[:, 1]
        train_scores.append(float(log_loss(labels[tr], model.predict_proba(fit_x)[:, 1])))
        print(f"seed {seed} fold {fold + 1}/4 done", flush=True)
    return {
        "raw_oof": round(float(log_loss(labels, oof)), 6),
        "pair_oof": round(float(log_loss(labels, adjust_pair(oof))), 6),
        "mean_fold_train": round(float(np.mean(train_scores)), 6),
    }, oof


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    dialogues = read_dialogues(ROOT / "data/train.json")
    numeric = train.select(FEATURES).to_numpy().astype(float)
    extended = np.array([
        new_features(dialogues[dialog_id], index)
        for dialog_id, index in zip(train["dialog_id"], train["participant_index"])
    ])
    if extended.shape[1] != len(NEW_FEATURES):
        raise ValueError("Unexpected feature count")
    labels = train["is_bot"].to_numpy()
    texts = train["text"].to_list()
    results = {}
    baseline = {}
    for seed in (23, 42):
        result, candidate_oof = run_seed(seed, numeric, extended, texts, labels)
        path = OUTPUTS / "v2/oof_predictions.parquet" if seed == 23 else OUTPUTS / "seed42/oof_predictions.parquet"
        frame = pl.read_parquet(path).sort(["dialog_id", "participant_index"])
        prior = frame["svd_catboost"].to_numpy()
        baseline[str(seed)] = {
            "raw_oof": round(float(log_loss(labels, prior)), 6),
            "pair_oof": round(float(log_loss(labels, adjust_pair(prior))), 6),
            "blend_pair_oof": round(float(log_loss(labels, frame["blend_pair"].to_numpy())), 6),
        }
        blend = .95 * (.25 * frame["cat_extended"].to_numpy() + .75 * candidate_oof)
        blend += .05 * frame["word_char_tfidf"].to_numpy()
        result["blend_pair_oof"] = round(float(log_loss(labels, adjust_pair(blend))), 6)
        results[str(seed)] = result
    report = {"new_features": NEW_FEATURES, "baseline": baseline, "candidate": results}
    (OUTPUTS / "analysis/candidate_features.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    mlflow.set_tracking_uri(f"sqlite:///{(OUTPUTS / 'mlflow.db').resolve().as_posix()}")
    mlflow.set_experiment("bot_detection")
    for seed, result in results.items():
        with mlflow.start_run(run_name=f"candidate_features_seed{seed}"):
            mlflow.log_params({"study": "candidate_features", "seed": seed, "new_feature_count": len(NEW_FEATURES)})
            mlflow.log_metric("svd_catboost_oof_logloss", result["pair_oof"])
            mlflow.log_metric("blend_pair_oof_logloss", result["blend_pair_oof"])
    print(json.dumps(report, ensure_ascii=False, indent=2))

main()


seed 23 fold 1/4 done
seed 23 fold 2/4 done
seed 23 fold 3/4 done
seed 23 fold 4/4 done
seed 42 fold 1/4 done
seed 42 fold 2/4 done
seed 42 fold 3/4 done
seed 42 fold 4/4 done
{
  "new_features": [
    "reply_overlap_high_share",
    "reply_novel_word_share",
    "reply_longer_share",
    "reply_prefix_extension_share",
    "reply_prefix_truncation_share",
    "reply_common_prefix_share",
    "reply_common_suffix_share",
    "last_reply_overlap",
    "last_reply_length_ratio",
    "question_to_question_share",
    "reply_script_switch_share"
  ],
  "baseline": {
    "23": {
      "raw_oof": 0.184413,
      "pair_oof": 0.180772,
      "blend_pair_oof": 0.177391
    },
    "42": {
      "raw_oof": 0.181779,
      "pair_oof": 0.177912,
      "blend_pair_oof": 0.174093
    }
  },
  "candidate": {
    "23": {
      "raw_oof": 0.186627,
      "pair_oof": 0.183563,
      "mean_fold_train": 0.046692,
      "blend_pair_oof": 0.17784
    },
    "42": {
      "raw_oof": 0.183,
      "pair_oof": 0

In [6]:
import json
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from main import FEATURES, adjust_pair, new_char_vectorizer, read_dialogues, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"
SEEDS = (23, 42)


def model(fold: int, l2: int = 4, iterations: int = 600) -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=iterations, depth=5, learning_rate=.04, l2_leaf_reg=l2,
        loss_function="Logloss", random_seed=42 + fold,
        thread_count=4, verbose=False,
    )


def scores(labels: np.ndarray, probabilities: np.ndarray, reference: pl.DataFrame) -> dict:
    blend = .95 * (.25 * reference["cat_extended"].to_numpy() + .75 * probabilities)
    blend += .05 * reference["word_char_tfidf"].to_numpy()
    return {
        "raw_oof": round(float(log_loss(labels, probabilities)), 6),
        "pair_oof": round(float(log_loss(labels, adjust_pair(probabilities))), 6),
        "blend_pair_oof": round(float(log_loss(labels, adjust_pair(blend))), 6),
    }


def run_seed(
    seed: int, numeric: np.ndarray, extended: np.ndarray,
    texts: list[str], labels: np.ndarray,
) -> dict:
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    cv = StratifiedKFold(4, shuffle=True, random_state=seed)
    predictions = {name: np.zeros(len(labels)) for name in NEW_FEATURES}
    predictions["bundle_l2_12_iter800"] = np.zeros(len(labels))
    train_losses = {name: [] for name in predictions}
    for fold, (train_dialogues, val_dialogues) in enumerate(
        cv.split(np.arange(len(pair_classes)), pair_classes)
    ):
        tr = row_indices(train_dialogues)
        va = row_indices(val_dialogues)
        vectorizer = new_char_vectorizer()
        char_train = vectorizer.fit_transform([texts[i] for i in tr])
        char_val = vectorizer.transform([texts[i] for i in va])
        svd = TruncatedSVD(n_components=48, n_iter=5, random_state=42 + fold)
        train_svd = svd.fit_transform(char_train)
        val_svd = svd.transform(char_val)
        for position, name in enumerate(NEW_FEATURES):
            fit_x = np.hstack([numeric[tr], extended[tr, position:position + 1], train_svd])
            val_x = np.hstack([numeric[va], extended[va, position:position + 1], val_svd])
            fitted = model(fold)
            fitted.fit(fit_x, labels[tr])
            predictions[name][va] = fitted.predict_proba(val_x)[:, 1]
            train_losses[name].append(float(log_loss(labels[tr], fitted.predict_proba(fit_x)[:, 1])))
            print(f"seed={seed} fold={fold + 1} feature={name}", flush=True)

        name = "bundle_l2_12_iter800"
        fit_x = np.hstack([numeric[tr], extended[tr], train_svd])
        val_x = np.hstack([numeric[va], extended[va], val_svd])
        fitted = model(fold, l2=12, iterations=800)
        fitted.fit(fit_x, labels[tr])
        predictions[name][va] = fitted.predict_proba(val_x)[:, 1]
        train_losses[name].append(float(log_loss(labels[tr], fitted.predict_proba(fit_x)[:, 1])))
        print(f"seed={seed} fold={fold + 1} feature={name}", flush=True)

    reference_path = OUTPUTS / "v2/oof_predictions.parquet" if seed == 23 else OUTPUTS / "seed42/oof_predictions.parquet"
    reference = pl.read_parquet(reference_path).sort(["dialog_id", "participant_index"])
    baseline = scores(labels, reference["svd_catboost"].to_numpy(), reference)
    scenarios = {}
    for name, probabilities in predictions.items():
        item = scores(labels, probabilities, reference)
        item["mean_fold_train"] = round(float(np.mean(train_losses[name])), 6)
        item["delta_vs_current_blend"] = round(item["blend_pair_oof"] - baseline["blend_pair_oof"], 6)
        scenarios[name] = item
    return {"baseline": baseline, "scenarios": scenarios}


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    dialogues = read_dialogues(ROOT / "data/train.json")
    numeric = train.select(FEATURES).to_numpy().astype(float)
    extended = np.array([
        new_features(dialogues[dialog_id], index)
        for dialog_id, index in zip(train["dialog_id"], train["participant_index"])
    ])
    labels = train["is_bot"].to_numpy()
    texts = train["text"].to_list()
    report = {"feature_names": NEW_FEATURES, "seeds": {}}
    path = OUTPUTS / "analysis/atomic_feature_study.json"
    for seed in SEEDS:
        report["seeds"][str(seed)] = run_seed(seed, numeric, extended, texts, labels)
        path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    baseline_alt = json.loads((OUTPUTS / "analysis/regularization_blend.json").read_text(encoding="utf-8"))
    for seed in SEEDS:
        item = report["seeds"][str(seed)]["scenarios"]["bundle_l2_12_iter800"]
        item["delta_vs_same_regularization_baseline"] = round(
            item["blend_pair_oof"] - baseline_alt[str(seed)]["stronger_l2_blend_oof"], 6
        )
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    database = (OUTPUTS / "mlflow.db").resolve()
    mlflow.set_tracking_uri(f"sqlite:///{database.as_posix()}")
    mlflow.set_experiment("bot_detection")
    for seed, item in report["seeds"].items():
        for name, result in item["scenarios"].items():
            with mlflow.start_run(run_name=f"atomic_{name}_seed{seed}"):
                mlflow.log_params({"study": "atomic_feature", "seed": seed, "feature": name})
                mlflow.log_metric("blend_pair_oof_logloss", result["blend_pair_oof"])
                mlflow.log_metric("delta_vs_current_blend", result["delta_vs_current_blend"])
                mlflow.log_metric("mean_fold_train_logloss", result["mean_fold_train"])
    print(json.dumps(report, ensure_ascii=False, indent=2))

main()


seed=23 fold=1 feature=reply_overlap_high_share
seed=23 fold=1 feature=reply_novel_word_share
seed=23 fold=1 feature=reply_longer_share
seed=23 fold=1 feature=reply_prefix_extension_share
seed=23 fold=1 feature=reply_prefix_truncation_share
seed=23 fold=1 feature=reply_common_prefix_share
seed=23 fold=1 feature=reply_common_suffix_share
seed=23 fold=1 feature=last_reply_overlap
seed=23 fold=1 feature=last_reply_length_ratio
seed=23 fold=1 feature=question_to_question_share
seed=23 fold=1 feature=reply_script_switch_share
seed=23 fold=1 feature=bundle_l2_12_iter800
seed=23 fold=2 feature=reply_overlap_high_share
seed=23 fold=2 feature=reply_novel_word_share
seed=23 fold=2 feature=reply_longer_share
seed=23 fold=2 feature=reply_prefix_extension_share
seed=23 fold=2 feature=reply_prefix_truncation_share
seed=23 fold=2 feature=reply_common_prefix_share
seed=23 fold=2 feature=reply_common_suffix_share
seed=23 fold=2 feature=last_reply_overlap
seed=23 fold=2 feature=last_reply_length_ratio
s

In [7]:
import json
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from main import FEATURES, new_char_vectorizer, read_dialogues, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"


def run_seed(seed: int, numeric: np.ndarray, extended: np.ndarray,
             texts: list[str], labels: np.ndarray) -> dict:
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    cv = StratifiedKFold(4, shuffle=True, random_state=seed)
    predictions = {f"{group}_{trees}": np.zeros(len(labels))
                   for group in ("baseline", "bundle") for trees in (600, 800)}
    train_losses = {name: [] for name in predictions}
    for fold, (train_dialogues, val_dialogues) in enumerate(
        cv.split(np.arange(len(pair_classes)), pair_classes)
    ):
        tr = row_indices(train_dialogues)
        va = row_indices(val_dialogues)
        vectorizer = new_char_vectorizer()
        char_train = vectorizer.fit_transform([texts[i] for i in tr])
        char_val = vectorizer.transform([texts[i] for i in va])
        svd = TruncatedSVD(n_components=48, n_iter=5, random_state=42 + fold)
        train_svd = svd.fit_transform(char_train)
        val_svd = svd.transform(char_val)
        for group, extra in (("baseline", False), ("bundle", True)):
            fit_x = np.hstack([numeric[tr], extended[tr], train_svd]) if extra else np.hstack([numeric[tr], train_svd])
            val_x = np.hstack([numeric[va], extended[va], val_svd]) if extra else np.hstack([numeric[va], val_svd])
            fitted = CatBoostClassifier(
                iterations=800, depth=5, learning_rate=.04, l2_leaf_reg=4,
                loss_function="Logloss", random_seed=42 + fold,
                thread_count=4, verbose=False,
            )
            fitted.fit(fit_x, labels[tr])
            for trees in (600, 800):
                name = f"{group}_{trees}"
                predictions[name][va] = fitted.predict_proba(val_x, ntree_end=trees)[:, 1]
                train_losses[name].append(float(log_loss(
                    labels[tr], fitted.predict_proba(fit_x, ntree_end=trees)[:, 1]
                )))
        print(f"seed={seed} fold={fold + 1}/4 done", flush=True)

    reference_path = OUTPUTS / "v2/oof_predictions.parquet" if seed == 23 else OUTPUTS / "seed42/oof_predictions.parquet"
    reference = pl.read_parquet(reference_path).sort(["dialog_id", "participant_index"])
    result = {}
    for name, probabilities in predictions.items():
        item = scores(labels, probabilities, reference)
        item["mean_fold_train"] = round(float(np.mean(train_losses[name])), 6)
        result[name] = item
    return result


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    dialogues = read_dialogues(ROOT / "data/train.json")
    numeric = train.select(FEATURES).to_numpy().astype(float)
    extended = np.array([
        new_features(dialogues[dialog_id], index)
        for dialog_id, index in zip(train["dialog_id"], train["participant_index"])
    ])
    texts = train["text"].to_list()
    labels = train["is_bot"].to_numpy()
    report = {str(seed): run_seed(seed, numeric, extended, texts, labels) for seed in (23, 42)}
    path = OUTPUTS / "analysis/bundle_iterations_study.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    mlflow.set_tracking_uri(f"sqlite:///{(OUTPUTS / 'mlflow.db').resolve().as_posix()}")
    mlflow.set_experiment("bot_detection")
    for seed, scenarios in report.items():
        for name, item in scenarios.items():
            with mlflow.start_run(run_name=f"trees_{name}_seed{seed}"):
                mlflow.log_params({"study": "bundle_iterations", "seed": seed, "scenario": name})
                mlflow.log_metric("blend_pair_oof_logloss", item["blend_pair_oof"])
                mlflow.log_metric("mean_fold_train_logloss", item["mean_fold_train"])
    print(json.dumps(report, ensure_ascii=False, indent=2))

main()


seed=23 fold=1/4 done
seed=23 fold=2/4 done
seed=23 fold=3/4 done
seed=23 fold=4/4 done
seed=42 fold=1/4 done
seed=42 fold=2/4 done
seed=42 fold=3/4 done
seed=42 fold=4/4 done
{
  "23": {
    "baseline_600": {
      "raw_oof": 0.184413,
      "pair_oof": 0.180772,
      "blend_pair_oof": 0.177391,
      "mean_fold_train": 0.048393
    },
    "baseline_800": {
      "raw_oof": 0.184021,
      "pair_oof": 0.181804,
      "blend_pair_oof": 0.175917,
      "mean_fold_train": 0.032592
    },
    "bundle_600": {
      "raw_oof": 0.186627,
      "pair_oof": 0.183563,
      "blend_pair_oof": 0.17784,
      "mean_fold_train": 0.046692
    },
    "bundle_800": {
      "raw_oof": 0.187078,
      "pair_oof": 0.185436,
      "blend_pair_oof": 0.176345,
      "mean_fold_train": 0.031107
    }
  },
  "42": {
    "baseline_600": {
      "raw_oof": 0.181779,
      "pair_oof": 0.177912,
      "blend_pair_oof": 0.174093,
      "mean_fold_train": 0.046093
    },
    "baseline_800": {
      "raw_oof": 0.18

Пакет из 11 новых признаков не дал устойчивого улучшения. Отдельная проверка признаков тоже не нашла варианта, который улучшал бы смесь на обоих разбиениях. Увеличение числа деревьев до 800 оказалось полезнее.


## Зависимость доли ботов от значений признаков


In [8]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.ticker import PercentFormatter


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"


def wilson_interval(positives: int, size: int) -> tuple[float, float]:
    z = 1.96
    rate = positives / size
    denominator = 1 + z * z / size
    center = (rate + z * z / (2 * size)) / denominator
    margin = z * np.sqrt(rate * (1 - rate) / size + z * z / (4 * size * size)) / denominator
    return center - margin, center + margin


def summarize(name: str, labels: list[str], groups: np.ndarray, target: np.ndarray) -> list[dict]:
    rows = []
    for code, label in enumerate(labels):
        values = target[groups == code]
        size = len(values)
        if size == 0:
            raise ValueError(f"Empty bin: {name} {label}")
        positives = int(values.sum())
        lower, upper = wilson_interval(positives, size)
        rows.append({
            "feature": name,
            "interval": label,
            "n": size,
            "bot_count": positives,
            "bot_share": positives / size,
            "ci_lower": lower,
            "ci_upper": upper,
        })
    return rows


def draw(ax, rows: list[dict], title: str, baseline: float, color: str, kind: str = "line") -> None:
    x = np.arange(len(rows))
    rates = np.array([row["bot_share"] for row in rows])
    lower = np.array([row["ci_lower"] for row in rows])
    upper = np.array([row["ci_upper"] for row in rows])
    errors = np.vstack([rates - lower, upper - rates])
    if kind == "bar":
        ax.bar(x, rates, width=.6, color=color, alpha=.85)
        ax.errorbar(x, rates, yerr=errors, fmt="none", ecolor="#253342", capsize=4, lw=1.4)
    else:
        ax.errorbar(x, rates, yerr=errors, fmt="o-", color=color, lw=2.6,
                    markersize=7, capsize=4, markeredgecolor="white", markeredgewidth=1.2)
    ax.axhline(baseline, color="#758394", lw=1.2, ls="--", zorder=0)
    for point, row in zip(x, rows):
        ax.text(point, min(row["ci_upper"] + .045, 1.085), f"n={row['n']}",
                ha="center", va="bottom", fontsize=9, color="#344054")
    ax.set_xticks(x, [row["interval"] for row in rows], fontsize=9)
    ax.set_ylim(0, 1.14)
    ax.set_yticks([0, .25, .5, .75, 1])
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.grid(axis="y", color="#e5eaf0", lw=.8)
    ax.set_axisbelow(True)
    ax.set_title(title, loc="left", fontsize=12, fontweight="bold", pad=13)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color("#d5dde6")


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    frame = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet")
    target = frame["is_bot"].to_numpy()
    baseline = float(target.mean())
    specs = [
        (
            "avg_prev_overlap", "Сходство ответа с предыдущей репликой",
            ["0", "(0; 0,25]", "(0,25; 0,5]", "(0,5; 0,75]", ">0,75"],
            np.digitize(frame["avg_prev_overlap"].to_numpy(), [0, .25, .5, .75], right=True),
            "#2558a5", "line",
        ),
        (
            "n_diff", "Разница в числе реплик",
            ["меньше", "поровну", "больше"],
            np.sign(frame["n_diff"].to_numpy()).astype(int) + 1,
            "#7037a6", "line",
        ),
        (
            "shared_vocab_ratio", "Общие слова с собеседником",
            ["0", "(0; 0,25]", "(0,25; 0,5]", "(0,5; 0,75]", ">0,75"],
            np.digitize(frame["shared_vocab_ratio"].to_numpy(), [0, .25, .5, .75], right=True),
            "#0d827a", "line",
        ),
        (
            "uppercase", "Доля заглавных букв",
            ["0", "(0; 5%]", "(5%; 15%]", ">15%"],
            np.digitize(frame["uppercase"].to_numpy(), [0, .05, .15], right=True),
            "#bc5b2b", "line",
        ),
        (
            "is_last_speaker", "Написал последнюю реплику",
            ["нет", "да"], frame["is_last_speaker"].to_numpy().astype(int),
            "#3c688c", "bar",
        ),
    ]

    plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": "white", "axes.facecolor": "white"})
    fig, axes = plt.subplots(2, 3, figsize=(17, 9.5), dpi=160)
    fig.suptitle("Как признаки связаны с долей ботов", x=.055, y=.975, ha="left",
                 fontsize=20, fontweight="bold", color="#172638")
    fig.text(.055, .935, "Train: 3032 участника. Точки показывают фактическую долю ботов в интервале; полосы - 95% интервал Уилсона.",
             fontsize=10.5, color="#4d5d70")
    rows = []
    for ax, (name, title, labels, groups, color, kind) in zip(axes.flat, specs):
        subset = summarize(name, labels, groups, target)
        rows.extend(subset)
        draw(ax, subset, title, baseline, color, kind)

    note = axes.flat[-1]
    note.axis("off")
    note.text(.02, .97, "Как читать график", va="top", fontsize=12,
              fontweight="bold", color="#172638", transform=note.transAxes)
    note.text(.02, .83,
              "Пунктир: средняя доля ботов, 40,8%.\n"
              "n: число участников в интервале.\n\n"
              "Сильнее всего растёт доля ботов при\n"
              "большем сходстве ответа и большем\n"
              "числе реплик. Заглавные буквы дают\n"
              "нелинейную картину.\n\n"
              "Это частоты в обучающей выборке,\n"
              "а не изменение вероятности для\n"
              "одного человека при прочих равных.",
              va="top", fontsize=11, linespacing=1.45, color="#344054", transform=note.transAxes)
    fig.subplots_adjust(left=.055, right=.985, top=.87, bottom=.09, hspace=.45, wspace=.25)
    fig.savefig(OUTPUTS / "analysis/feature_effects.png", facecolor="white", bbox_inches="tight")
    plt.close(fig)
    pl.DataFrame(rows).write_csv(OUTPUTS / "analysis/feature_effects_bins.csv")
    print({name: [(row["interval"], row["n"], round(row["bot_share"], 3))
                  for row in rows if row["feature"] == name] for name, *_ in specs})

main()


{'avg_prev_overlap': [('0', 1610, 0.207), ('(0; 0,25]', 664, 0.446), ('(0,25; 0,5]', 226, 0.584), ('(0,5; 0,75]', 103, 0.845), ('>0,75', 429, 0.904)], 'n_diff': [('меньше', 991, 0.045), ('поровну', 1050, 0.371), ('больше', 991, 0.809)], 'shared_vocab_ratio': [('0', 866, 0.327), ('(0; 0,25]', 1094, 0.407), ('(0,25; 0,5]', 356, 0.447), ('(0,5; 0,75]', 180, 0.483), ('>0,75', 536, 0.491)], 'uppercase': [('0', 1563, 0.454), ('(0; 5%]', 510, 0.547), ('(5%; 15%]', 748, 0.257), ('>15%', 211, 0.27)], 'is_last_speaker': [('нет', 1516, 0.086), ('да', 1516, 0.73)]}


![Зависимость доли ботов от значений признаков](../outputs/analysis/feature_effects.png)

Точки показывают долю ботов внутри интервалов признака. Для редких интервалов неопределённость шире, поэтому на них нельзя опираться как на универсальное правило.
